In [1]:
# Comparaison de bits entre images avec Jupyter Notebook

# Installation des dépendances (à exécuter une seule fois)

# Importations nécessaires
from PIL import Image
import numpy as np
import scapy.all as scapy
import os

def compare_image_bits(image1_path, image2_path):
    """
    Compare deux images bit par bit et génère un rapport détaillé.
    
    Args:
        image1_path (str): Chemin de la première image
        image2_path (str): Chemin de la deuxième image
    
    Returns:
        dict: Dictionnaire contenant les informations de comparaison
    """
    # Vérifier que les fichiers existent
    if not os.path.exists(image1_path) or not os.path.exists(image2_path):
        raise FileNotFoundError("L'un des fichiers image n'existe pas")
    
    # Ouvrir les images
    img1 = Image.open(image1_path)
    img2 = Image.open(image2_path)
    
    # Convertir les images en tableaux numpy
    arr1 = np.array(img1)
    arr2 = np.array(img2)
    
    # Vérifier les dimensions
    if arr1.shape != arr2.shape:
        raise ValueError("Les images ont des dimensions différentes")
    
    # Comparaison bit par bit
    diff_mask = arr1 != arr2
    total_pixels = arr1.size
    different_pixels = np.sum(diff_mask)
    difference_percentage = (different_pixels / total_pixels) * 100
    
    # Génération du rapport détaillé
    rapport = {
        'dimensions': {
            'largeur': arr1.shape[1],
            'hauteur': arr1.shape[0],
            'canaux': arr1.shape[2] if len(arr1.shape) > 2 else 1
        },
        'comparaison': {
            'pixels_total': total_pixels,
            'pixels_differents': int(different_pixels),
            'pourcentage_difference': round(difference_percentage, 4)
        },
        'differences_par_canal': {}
    }
    
    # Analyse des différences par canal pour les images couleur
    if len(arr1.shape) > 2:
        for canal in range(arr1.shape[2]):
            canal_diff = np.sum(arr1[:,:,canal] != arr2[:,:,canal])
            rapport['differences_par_canal'][f'canal_{canal}'] = int(canal_diff)
    
    return rapport

def generer_rapport_hexadecimal(image1_path, image2_path):
    """
    Génère un rapport hexadécimal des différences entre deux images.
    
    Args:
        image1_path (str): Chemin de la première image
        image2_path (str): Chemin de la deuxième image
    
    Returns:
        list: Liste des paquets hexadécimaux représentant les différences
    """
    with open(image1_path, 'rb') as f1, open(image2_path, 'rb') as f2:
        contenu1 = f1.read()
        contenu2 = f2.read()
    
    # Générer des paquets hexadécimaux pour les différences
    differences = []
    for i in range(min(len(contenu1), len(contenu2))):
        if contenu1[i] != contenu2[i]:
            paquet = scapy.IP(
                src=f"0.{contenu1[i]}.{contenu2[i]}.255",
                dst=f"255.{i}.{hex(contenu1[i])[2:].zfill(2)}.{hex(contenu2[i])[2:].zfill(2)}"
            )
            differences.append(paquet)
    
    return differences

# Visualisation des différences (optionnel)
def visualiser_differences(image1_path, image2_path):
    """
    Crée une image de différence entre deux images.
    
    Args:
        image1_path (str): Chemin de la première image
        image2_path (str): Chemin de la deuxième image
    
    Returns:
        numpy.ndarray: Image des différences
    """
    img1 = Image.open(image1_path)
    img2 = Image.open(image2_path)
    
    arr1 = np.array(img1)
    arr2 = np.array(img2)
    
    # Crée un masque de différence
    diff_mask = arr1 != arr2
    
    # Crée une image de différence
    diff_image = np.zeros_like(arr1)
    diff_image[diff_mask] = [255, 0, 0]  # Marque les différences en rouge
    
    return diff_image

# Exemple d'utilisation dans le notebook
# Remplacez 'chemin/vers/image1.png' et 'chemin/vers/image2.png' par vos chemins d'images
image1 = 'chemin/vers/image1.png'
image2 = 'chemin/vers/image2.png'

# Comparaison des bits
rapport_bits = compare_image_bits(image1, image2)
print("Rapport de comparaison des bits :")
print(rapport_bits)

# Génération du rapport hexadécimal
differences_hex = generer_rapport_hexadecimal(image1, image2)
print(f"\nNombre de différences hexadécimales : {len(differences_hex)}")

# Visualisation des différences (optionnel)
import matplotlib.pyplot as plt

diff_image = visualiser_differences(image1, image2)
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.title('Image 1')
plt.imshow(Image.open(image1))
plt.subplot(1, 2, 2)
plt.title('Différences')
plt.imshow(diff_image)
plt.show()

ModuleNotFoundError: No module named 'scapy'